In [1]:
import sys
import json
from pathlib import Path

import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import openpyxl
from PIL import Image, ImageOps

print("Python location:", sys.executable)
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("Notebook packages loaded.")

Python location: /opt/anaconda3/envs/blv-project/bin/python
NumPy: 2.1.3
OpenCV: 5.0.0
Notebook packages loaded.


In [2]:
# Find the project folder
current_folder = Path.cwd().resolve()

PROJECT_FOLDER = next(
    (
        folder
        for folder in [current_folder, *current_folder.parents]
        if (folder / ".git").exists()
        and (folder / "sampling" / "outputs").is_dir()
    ),
    None
)

if PROJECT_FOLDER is None:
    raise FileNotFoundError(
        "Open the blv-project folder in VS Code, then try again."
    )

OUTPUT_FOLDER = PROJECT_FOLDER / "sampling" / "outputs"
FINAL_FOLDER = OUTPUT_FOLDER / "final_sample"
PILOT_FOLDER = OUTPUT_FOLDER / "blur_calibration"

# Read the existing sample and blur scores
sample_file = FINAL_FOLDER / "sample_manifest_with_blur.csv"
sample = pd.read_csv(sample_file)

# Check the saved records
expected_groups = {
    "blurry": 70,
    "clear": 70,
    "low_light": 60,
    "text": 50,
}

assert len(sample) == 250, "Expected 250 image records."
assert sample["file_name"].notna().all(), "Missing filenames."
assert sample["file_name"].is_unique, "Duplicate filenames found."
assert sample["laplacian_variance"].notna().all(), "Missing blur scores."
assert sample["stratum"].value_counts().to_dict() == expected_groups, (
    "The group counts do not match the agreed sample."
)

print("Project folder:", PROJECT_FOLDER)
print("Selected images:", len(sample))
print("Images with blur scores:", sample["laplacian_variance"].notna().sum())
print("\nGroup counts:")
print(sample["stratum"].value_counts())

Project folder: /Users/a/Desktop/blv-project-team
Selected images: 250
Images with blur scores: 250

Group counts:
stratum
blurry       70
clear        70
low_light    60
text         50
Name: count, dtype: int64


In [3]:
# Read the existing pilot scores and selected threshold
pilot = pd.read_csv(
    PILOT_FOLDER / "pilot_with_blur_scores.csv"
)

with open(PILOT_FOLDER / "selected_threshold.json") as file:
    threshold_settings = json.load(file)

# Check the pilot records
assert len(pilot) == 200, "Expected 200 pilot images."
assert pilot["file_name"].notna().all(), "Missing pilot filenames."
assert pilot["file_name"].is_unique, "Duplicate pilot images found."
assert pilot["laplacian_variance"].notna().all(), "Missing pilot blur scores."

# Check that the pilot and final sample contain different images
overlap = set(pilot["file_name"]) & set(sample["file_name"])
assert not overlap, f"Found {len(overlap)} overlapping images."

# Check the calibration and evaluation groups
expected_pilot_groups = {
    ("choose_threshold", "blurry"): 50,
    ("choose_threshold", "clear"): 50,
    ("check_threshold", "blurry"): 50,
    ("check_threshold", "clear"): 50,
}

actual_pilot_groups = (
    pilot.groupby(["purpose", "reference_group"]).size().to_dict()
)

assert actual_pilot_groups == expected_pilot_groups, (
    "The pilot groups do not match the expected split."
)

threshold = threshold_settings["threshold"]

print("Pilot images:", len(pilot))
print("Pilot images with blur scores:", pilot["laplacian_variance"].notna().sum())
print("Overlap with final sample:", len(overlap))
print("Saved threshold:", threshold)
print("\nPilot groups:")
print(pd.crosstab(pilot["purpose"], pilot["reference_group"]))

Pilot images: 200
Pilot images with blur scores: 200
Overlap with final sample: 0
Saved threshold: 164.67810814512066

Pilot groups:
reference_group   blurry  clear
purpose                        
check_threshold       50     50
choose_threshold      50     50


In [4]:
# Current location of the image folders on your Mac
IMAGE_ROOT = Path.home() / "Desktop" / "blv-project" / "raw_data"

# Combine the final sample and pilot image lists
required_images = pd.concat(
    [
        sample[["file_name", "split"]],
        pilot[["file_name", "split"]],
    ],
    ignore_index=True,
)

image_problems = []

for _, row in required_images.iterrows():
    image_path = IMAGE_ROOT / row["split"] / row["file_name"]

    try:
        with Image.open(image_path) as image:
            image.load()
    except Exception as error:
        image_problems.append({
            "file_name": row["file_name"],
            "problem": str(error),
        })

print("Image location:", IMAGE_ROOT)
print("Images checked:", len(required_images))
print("Images successfully opened:", len(required_images) - len(image_problems))
print("Images needing attention:", len(image_problems))

if image_problems:
    display(pd.DataFrame(image_problems))

Image location: /Users/a/Desktop/blv-project/raw_data
Images checked: 450
Images successfully opened: 450
Images needing attention: 0
